In [ ]:
import pandas as pd, numpy as np
import vivarium_inputs
import gbd_mapping
import pathlib

In [ ]:
location = "India"
vehicle = "rice"

In [ ]:
location = location.title()

In [ ]:
pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

In [ ]:
asfr = vivarium_inputs.get_measure(gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location).value
asfr[asfr > 0]

In [ ]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel("parameter")
asfr

In [ ]:
births = (pop * asfr)
births[births > 0]

In [ ]:
births = births.sum()
f'{int(births):,}'

In [ ]:
sim_baseline_births = pd.read_parquet(f"../../0200_pregnancy_sim/sim_results/{vehicle}/{location.lower()}/pregnancy_outcome_count.parquet")
sim_baseline_births

In [ ]:
sim_baseline_births = sim_baseline_births[
    (sim_baseline_births.scenario == 'baseline') &
    (sim_baseline_births.sub_entity == 'live_birth')
]
sim_baseline_births

In [ ]:
sim_baseline_births = sim_baseline_births.groupby("input_draw").value.sum().mean()
sim_baseline_births

In [ ]:
births

In [ ]:
scalar = births / sim_baseline_births
scalar

In [ ]:
for result in ["ylds", "ylls", "deaths", "person_time"]:
    df = pd.read_parquet(f"../../0300_child_sim/sim_results/{vehicle}/{location.lower()}/{result}.parquet")
    df.value *= scalar
    path = pathlib.Path(f'../results/rescaled_child_results/{vehicle}/{location.lower()}/{result}.parquet')
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)